# Stream-Stream Join with Watermarking: A Step-by-Step Demo (Scala)

This notebook demonstrates how **watermarking** and **range conditions** work together in Spark Structured Streaming stream-stream joins.

## Scenario
We have two streams:
- **Ad Impressions** — when an ad is shown to a user
- **Ad Clicks** — when a user clicks on an ad

We want to join clicks to impressions, but only if the click happened **within 10 minutes** of the impression (the range condition).

## Key Concepts

| Concept | What it does |
|---|---|
| **Watermark** | Tells Spark how late data can arrive. Defined as `max(event_time) - threshold`. Data older than the watermark is dropped. |
| **Range condition** | Restricts the join to matching rows within a time window (e.g., click within 10 min of impression). |
| **Together** | The watermark + range condition let Spark know when it's safe to evict old state — without them, state grows unboundedly. |

## What We'll Show
| Run | Scenario | Expected Result |
|---|---|---|
| 1 | On-time data, all within range | All pairs match |
| 2 | Clicks at the edge of the 10-min range | In-range clicks match; out-of-range click is excluded |
| 3 | Late data arriving after the watermark | Dropped — no new matches |
| 4 | Late-ish data still within the watermark | Accepted — produces a match |
| 5 | Advance watermark to trigger state eviction | clk_5 evicted from state store |

## Setup
Create the Delta tables and define paths. Running this cell resets all state so the demo is repeatable.

In [0]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import org.apache.spark.sql.Row
import java.sql.Timestamp
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val demoCatalog = "alexn"
val demoSchema = "wmdemo"

spark.sql(s"CREATE SCHEMA IF NOT EXISTS $demoCatalog.$demoSchema")

val impressionsTable = s"$demoCatalog.$demoSchema.impressions_scala"
val clicksTable = s"$demoCatalog.$demoSchema.clicks_scala"
val outputTable = s"$demoCatalog.$demoSchema.matched_clicks_scala"
val checkpointPath = "/Volumes/alexn/default/v/checkpoints/wmdemo_scala"

// --- Clean up prior runs ---
spark.sql(s"DROP TABLE IF EXISTS $outputTable")
spark.sql(s"DROP TABLE IF EXISTS $impressionsTable")
spark.sql(s"DROP TABLE IF EXISTS $clicksTable")
dbutils.fs.rm(checkpointPath, recurse = true)

// --- Create empty Delta tables with schema ---
val impressionsSchema = StructType(Seq(
  StructField("impression_id", StringType),
  StructField("ad_id", StringType),
  StructField("impression_time", TimestampType)
))

val clicksSchema = StructType(Seq(
  StructField("click_id", StringType),
  StructField("ad_id", StringType),
  StructField("click_time", TimestampType)
))

spark.createDataFrame(java.util.Collections.emptyList[Row](), impressionsSchema).write.format("delta").saveAsTable(impressionsTable)
spark.createDataFrame(java.util.Collections.emptyList[Row](), clicksSchema).write.format("delta").saveAsTable(clicksTable)

println(s"Schema:      $demoCatalog.$demoSchema")
println(s"Impressions: $impressionsTable")
println(s"Clicks:      $clicksTable")
println(s"Output:      $outputTable")
println(s"Checkpoint:  $checkpointPath")
println("Setup complete.")

Schema:      alexn.wmdemo
Impressions: alexn.wmdemo.impressions_scala
Clicks:      alexn.wmdemo.clicks_scala
Output:      alexn.wmdemo.matched_clicks_scala
Checkpoint:  /Volumes/alexn/default/v/checkpoints/wmdemo_scala
Setup complete.


import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import org.apache.spark.sql.Row
import java.sql.Timestamp
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter
demoCatalog: String = "alexn"
demoSchema: String = "wmdemo"
res6_8: org.apache.spark.sql.package.DataFrame = []
impressionsTable: String = "alexn.wmdemo.impressions_scala"
clicksTable: String = "alexn.wmdemo.clicks_scala"
outputTable: String = "alexn.wmdemo.matched_clicks_scala"
checkpointPath: String = "/Volumes/alexn/default/v/checkpoints/wmdemo_scala"
res6_13: org.apache.spark.sql.package.DataFrame = []
res6_14: org.apache.spark.sql.package.DataFrame = []
res6_15: org.apache.spark.sql.package.DataFrame = []
res6_16: Boolean = true
impressionsSchema: StructType = Seq(
  StructField(
    name = "impression_id",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "ad_id",
    dataType = StringType,
    nullable = true,
    metadata = {}
 

## Helper Functions

In [0]:
import org.apache.spark.sql.streaming.StreamingQueryListener
import org.apache.spark.sql.streaming.StreamingQueryListener._
import scala.collection.JavaConverters._
import scala.util.Try

val fmt = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm")

def ts(timeStr: String): Timestamp =
  Timestamp.valueOf(LocalDateTime.parse(s"2025-01-15 $timeStr", fmt))

def writeImpressions(rows: Seq[(String, String, String)]): Unit = {
  val data = rows.map { case (id, adId, t) => Row(id, adId, ts(t)) }
  val df = spark.createDataFrame(data.asJava, impressionsSchema)
  df.write.format("delta").mode("append").saveAsTable(impressionsTable)
  println(s"Wrote ${rows.size} impressions:")
  rows.foreach { case (id, adId, t) => println(f"  $id%-8s  ad=$adId%-5s  time=$t") }
}

def writeClicks(rows: Seq[(String, String, String)]): Unit = {
  val data = rows.map { case (id, adId, t) => Row(id, adId, ts(t)) }
  val df = spark.createDataFrame(data.asJava, clicksSchema)
  df.write.format("delta").mode("append").saveAsTable(clicksTable)
  println(s"Wrote ${rows.size} clicks:")
  rows.foreach { case (id, adId, t) => println(f"  $id%-8s  ad=$adId%-5s  time=$t") }
}

// --- Listener to capture watermark metrics ---
val watermarkListener = new StreamingQueryListener {
  override def onQueryStarted(event: QueryStartedEvent): Unit =
    println(s"Query started: ${event.id}")

  override def onQueryProgress(event: QueryProgressEvent): Unit = {
    val progress = event.progress
    val eventTime = progress.eventTime
    if (eventTime != null) {
      val watermark = eventTime.getOrDefault("watermark", null)
      if (watermark != null) println(s"  eventTime.watermark: $watermark")
    }
    progress.stateOperators.foreach { op =>
      println(s"  stateOp: ${op.operatorName} | rows_updated=${op.numRowsUpdated}, rows_removed=${op.numRowsRemoved}, dropped_by_watermark=${op.numRowsDroppedByWatermark}, rows_total=${op.numRowsTotal}")
    }
  }

  override def onQueryTerminated(event: QueryTerminatedEvent): Unit =
    println(s"Query terminated: ${event.id}")
}

spark.streams.addListener(watermarkListener)

def runStreamingJoin(): Unit = {
  val impressionsStream = spark.readStream
    .format("delta")
    .table(impressionsTable)
    .withWatermark("impression_time", "5 minutes")

  val clicksStream = spark.readStream
    .format("delta")
    .table(clicksTable)
    .withWatermark("click_time", "5 minutes")

  val joinCondition =
    impressionsStream("ad_id") === clicksStream("ad_id") &&
    clicksStream("click_time") >= impressionsStream("impression_time") &&
    clicksStream("click_time") <= impressionsStream("impression_time") + expr("INTERVAL 10 MINUTES")

  val joined = impressionsStream.join(clicksStream, joinCondition, "inner")

  val query = joined
    .select(
      impressionsStream("impression_id"),
      clicksStream("click_id"),
      impressionsStream("ad_id"),
      impressionsStream("impression_time"),
      clicksStream("click_time"),
      (unix_timestamp(col("click_time")) - unix_timestamp(col("impression_time"))).cast("int").alias("delay_seconds")
    )
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpointPath)
    .trigger(org.apache.spark.sql.streaming.Trigger.AvailableNow())
    .toTable(outputTable)

  query.awaitTermination()
  println("Streaming join complete.")
}

def showResults(): Unit = {
  display(spark.table(outputTable).orderBy("impression_time", "click_time"))
}

def showGlobalWatermarks(): Unit = {
  import java.time.{Instant, ZoneOffset}

  val commitsPath = s"$checkpointPath/commits"
  val commitFiles = dbutils.fs.ls(commitsPath).filter(f => Try(f.name.trim.toInt).isSuccess)
  val wmPattern = """"nextBatchWatermarkMs":(\d+)""".r

  val rows = commitFiles.sortBy(f => f.name.trim.toInt).flatMap { f =>
    val content = dbutils.fs.head(f.path)
    val lastLine = content.trim.split("\n").last
    wmPattern.findFirstMatchIn(lastLine).map { m =>
      val wmMs = m.group(1).toLong
      val wmTs = Instant.ofEpochMilli(wmMs).atZone(ZoneOffset.UTC).toString
      Row(f.name.trim.toInt, wmMs, wmTs)
    }
  }

  val wmSchema = StructType(Seq(
    StructField("batch_id", IntegerType),
    StructField("watermark_epoch_ms", LongType),
    StructField("watermark_timestamp", StringType)
  ))

  display(spark.createDataFrame(rows.asJava, wmSchema))
}

3 deprecations (since 2.13.0); re-run enabling -deprecation for details, or try -help
26/04/05 21:42:04 INFO StreamingQueryListenerBus: Added streaming query listener: ammonite.$sess.cmd7$Helper$$anon$1
26/04/05 21:42:04 INFO StreamingQueryListenerBus: Registered server side listener successfully


import org.apache.spark.sql.streaming.StreamingQueryListener
import org.apache.spark.sql.streaming.StreamingQueryListener._
import scala.collection.JavaConverters._
import scala.util.Try
fmt: DateTimeFormatter = Value(YearOfEra,4,19,EXCEEDS_PAD)'-'Value(MonthOfYear,2)'-'Value(DayOfMonth,2)' 'Value(HourOfDay,2)':'Value(MinuteOfHour,2)
defined function ts
defined function writeImpressions
defined function writeClicks
watermarkListener: StreamingQueryListener = ammonite.$sess.cmd7$Helper$$anon$1@78ee3be2
defined function runStreamingJoin
defined function showResults
defined function showGlobalWatermarks

---
## Run 1: On-Time Data — Everything Matches

We write 3 impressions and 3 corresponding clicks. Every click happens within a few minutes of its impression — well within the **10-minute range** and the **5-minute watermark**.

| Impression | Time | Click | Time | Delay |
|---|---|---|---|---|
| imp_1 (ad_A) | 12:00 | clk_1 (ad_A) | 12:03 | 3 min |
| imp_2 (ad_B) | 12:02 | clk_2 (ad_B) | 12:06 | 4 min |
| imp_3 (ad_C) | 12:05 | clk_3 (ad_C) | 12:08 | 3 min |

**Expected: all 3 pairs match.**

In [0]:
writeImpressions(Seq(
  ("imp_1", "ad_A", "12:00"),
  ("imp_2", "ad_B", "12:02"),
  ("imp_3", "ad_C", "12:05")
))

writeClicks(Seq(
  ("clk_1", "ad_A", "12:03"),
  ("clk_2", "ad_B", "12:06"),
  ("clk_3", "ad_C", "12:08")
))

runStreamingJoin()

Wrote 3 impressions:
  imp_1     ad=ad_A   time=12:00
  imp_2     ad=ad_B   time=12:02
  imp_3     ad=ad_C   time=12:05
Wrote 3 clicks:
  clk_1     ad=ad_A   time=12:03
  clk_2     ad=ad_B   time=12:06
  clk_3     ad=ad_C   time=12:08


26/04/05 21:42:09 WARN Column: Constructing trivially true equals predicate, 'ad_id == ad_id'. Perhaps you need to use aliases.
26/04/05 21:42:10 INFO DatabricksEdgeConfigs: serverlessEnabled : false
26/04/05 21:42:10 INFO DatabricksEdgeConfigs: perfPackEnabled : true
26/04/05 21:42:10 INFO DatabricksEdgeConfigs: classicSqlEnabled : true
26/04/05 21:42:10 INFO Log4jUsageLogger: clusterRuntimeMode=1.0, tags=List(classicSqlEnabled=true, perfPackEnabled=true, serverlessEnabled=false), blob=null
26/04/05 21:42:10 INFO DatabricksEdgeConfigs: spark.databricks.test.default.enabled : false
26/04/05 21:42:11 INFO AbstractParser$ParserCaches: EXPERIMENTAL: Query cached 5 DFA states in the parser. Total cached DFA states: 5in the parser. Driver memory: 22649241600.
26/04/05 21:42:14 INFO StreamingQueryListenerBus: Received QueryStartedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


Query started: 1120a40b-a62c-4b4d-993a-44174cbd215b


26/04/05 21:42:14 INFO DataStreamWriter: Called streaming start call back for query: 1120a40b-a62c-4b4d-993a-44174cbd215b


  eventTime.watermark: 1970-01-01T00:00:00.000Z
  stateOp: symmetricHashJoin | rows_updated=6, rows_removed=0, dropped_by_watermark=0, rows_total=6
  eventTime.watermark: 2025-01-15T12:00:00.000Z
  stateOp: symmetricHashJoin | rows_updated=0, rows_removed=0, dropped_by_watermark=0, rows_total=6
Streaming join complete.


26/04/05 21:44:43 INFO StreamingQueryListenerBus: Received QueryTerminatedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


NOTE: Two events (microbatches) are shown because availableNow=True trigger always ends with an extra empty ("no-data") batch.


[source](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)

### Run 1 Results
All 3 pairs should appear. After this batch:
- **Impression watermark** = max(12:05) − 5 min = **12:00**
- **Click watermark** = max(12:08) − 5 min = **12:03**

In [0]:
showResults()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180


In [0]:
display(spark.read.format("statestore").option("joinSide", "left").load(checkpointPath))

key,value,partition_id
List(ad_A),"List(imp_1, ad_A, 2025-01-15T12:00:00Z)",90
List(ad_B),"List(imp_2, ad_B, 2025-01-15T12:02:00Z)",143
List(ad_C),"List(imp_3, ad_C, 2025-01-15T12:05:00Z)",197


In [0]:
display(spark.read.format("statestore").option("joinSide", "right").load(checkpointPath))

key,value,partition_id
List(ad_A),"List(clk_1, ad_A, 2025-01-15T12:03:00Z)",90
List(ad_B),"List(clk_2, ad_B, 2025-01-15T12:06:00Z)",143
List(ad_C),"List(clk_3, ad_C, 2025-01-15T12:08:00Z)",197


In [0]:
showGlobalWatermarks()

batch_id,watermark_epoch_ms,watermark_timestamp
0,1736942400000,2025-01-15T12:00Z
1,1736942400000,2025-01-15T12:00Z


---
## Run 2: Range Condition Boundary

Now we test the **10-minute range condition**. We write 2 new impressions and 3 clicks:

| Impression | Time | Click | Time | Delay | Within 10-min range? |
|---|---|---|---|---|---|
| imp_4 (ad_D) | 12:20 | clk_4 (ad_D) | 12:29 | 9 min | **Yes** |
| imp_4 (ad_D) | 12:20 | clk_5 (ad_D) | 12:31 | 11 min | **No — too late** |
| imp_5 (ad_E) | 12:22 | clk_6 (ad_E) | 12:25 | 3 min | **Yes** |

`clk_5` is for the same ad as `imp_4`, but it arrives **11 minutes** after the impression — outside the range.

**Expected: 2 new matches (clk_4 + clk_6). clk_5 is excluded by the range condition.
It does not show up in the output (but is retained in the state store for now)**

In [0]:
writeImpressions(Seq(
  ("imp_4", "ad_D", "12:20"),
  ("imp_5", "ad_E", "12:22")
))

writeClicks(Seq(
  ("clk_4", "ad_D", "12:29"),
  ("clk_5", "ad_D", "12:31"),
  ("clk_6", "ad_E", "12:25")
))

runStreamingJoin()

Wrote 2 impressions:
  imp_4     ad=ad_D   time=12:20
  imp_5     ad=ad_E   time=12:22
Wrote 3 clicks:
  clk_4     ad=ad_D   time=12:29
  clk_5     ad=ad_D   time=12:31
  clk_6     ad=ad_E   time=12:25


26/04/05 21:46:37 WARN Column: Constructing trivially true equals predicate, 'ad_id == ad_id'. Perhaps you need to use aliases.
26/04/05 21:46:37 INFO AbstractParser$ParserCaches: EXPERIMENTAL: Query cached 0 DFA states in the parser. Total cached DFA states: 5in the parser. Driver memory: 22649241600.
26/04/05 21:46:38 INFO StreamingQueryListenerBus: Received QueryStartedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


Query started: 1120a40b-a62c-4b4d-993a-44174cbd215b


26/04/05 21:46:38 INFO DataStreamWriter: Called streaming start call back for query: 1120a40b-a62c-4b4d-993a-44174cbd215b


  eventTime.watermark: 2025-01-15T12:00:00.000Z
  stateOp: symmetricHashJoin | rows_updated=5, rows_removed=0, dropped_by_watermark=0, rows_total=11
  eventTime.watermark: 2025-01-15T12:17:00.000Z
  stateOp: symmetricHashJoin | rows_updated=0, rows_removed=6, dropped_by_watermark=0, rows_total=5


26/04/05 21:47:18 INFO StreamingQueryListenerBus: Received QueryTerminatedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


Streaming join complete.
Query terminated: 1120a40b-a62c-4b4d-993a-44174cbd215b


NOTE: 6 rows are evicted from the state store because they can no longer participate in future joins based on the combination of watermark and range condition.

### Run 2 Results
You should see **5 total rows** (3 from Run 1 + 2 new). `clk_5` at 12:31 is absent — it was 11 minutes after `imp_4`, exceeding the 10-minute range.

Updated watermarks:
- **Impression watermark** = max(12:22) − 5 min = **12:17**
- **Click watermark** = max(12:31) − 5 min = **12:26**

In [0]:
showResults()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180
imp_4,clk_4,ad_D,2025-01-15T12:20:00Z,2025-01-15T12:29:00Z,540
imp_5,clk_6,ad_E,2025-01-15T12:22:00Z,2025-01-15T12:25:00Z,180


In [0]:
display(spark.read.format("statestore").option("joinSide", "left").load(checkpointPath))

key,value,partition_id
List(ad_D),"List(imp_4, ad_D, 2025-01-15T12:20:00Z)",5
List(ad_E),"List(imp_5, ad_E, 2025-01-15T12:22:00Z)",67


In [0]:
display(spark.read.format("statestore").option("joinSide", "right").load(checkpointPath))

key,value,partition_id
List(ad_D),"List(clk_4, ad_D, 2025-01-15T12:29:00Z)",5
List(ad_D),"List(clk_5, ad_D, 2025-01-15T12:31:00Z)",5
List(ad_E),"List(clk_6, ad_E, 2025-01-15T12:25:00Z)",67


In [0]:
showGlobalWatermarks()

batch_id,watermark_epoch_ms,watermark_timestamp
0,1736942400000,2025-01-15T12:00Z
1,1736942400000,2025-01-15T12:00Z
2,1736943420000,2025-01-15T12:17Z
3,1736943420000,2025-01-15T12:17Z


---
## Run 3: Late Data — Dropped by Watermark

This is the key watermarking test. We write an impression at **12:10** and a click at **12:12** — data that "arrived late."

But the watermarks have already advanced past these times:
- Impression watermark is **12:17** → an impression at 12:10 is **behind** the watermark
- Click watermark is **12:26** → a click at 12:12 is **behind** the watermark

Spark drops both records because they are too late.

**Expected: no new matches. The output table stays at 5 rows.**

In [0]:
writeImpressions(Seq(
  ("imp_6", "ad_F", "12:10")
))

writeClicks(Seq(
  ("clk_7", "ad_F", "12:12")
))

runStreamingJoin()

Wrote 1 impressions:
  imp_6     ad=ad_F   time=12:10
Wrote 1 clicks:
  clk_7     ad=ad_F   time=12:12


26/04/05 21:49:15 WARN Column: Constructing trivially true equals predicate, 'ad_id == ad_id'. Perhaps you need to use aliases.
26/04/05 21:49:15 INFO AbstractParser$ParserCaches: EXPERIMENTAL: Query cached 0 DFA states in the parser. Total cached DFA states: 5in the parser. Driver memory: 22649241600.
26/04/05 21:49:16 INFO StreamingQueryListenerBus: Received QueryStartedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


Query started: 1120a40b-a62c-4b4d-993a-44174cbd215b


26/04/05 21:49:16 INFO DataStreamWriter: Called streaming start call back for query: 1120a40b-a62c-4b4d-993a-44174cbd215b


  eventTime.watermark: 2025-01-15T12:17:00.000Z
  stateOp: symmetricHashJoin | rows_updated=0, rows_removed=0, dropped_by_watermark=2, rows_total=5


26/04/05 21:49:56 INFO StreamingQueryListenerBus: Received QueryTerminatedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


Query terminated: 1120a40b-a62c-4b4d-993a-44174cbd215b
Streaming join complete.


### Run 3 Results
Still **5 rows** — the late data was silently dropped. This is the watermark in action: it protects the system from unbounded state growth by discarding data that arrives too late to matter.

In [0]:
showResults()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180
imp_4,clk_4,ad_D,2025-01-15T12:20:00Z,2025-01-15T12:29:00Z,540
imp_5,clk_6,ad_E,2025-01-15T12:22:00Z,2025-01-15T12:25:00Z,180


---
## Run 4: Late-ish Data — Still Within Watermark

Not all "late" data gets dropped. We write an impression at **12:21** and a click at **12:27**.

Checking against the watermarks:
- Impression at 12:21 > impression watermark of **12:17** → **accepted**
- Click at 12:27 > click watermark of **12:26** → **accepted**
- Delay = 6 minutes → **within the 10-minute range**

Even though this data arrives in a later micro-batch than Run 2, its event times are still within the watermark threshold, so Spark processes it normally.

**Expected: 1 new match → 6 total rows.**

In [0]:
writeImpressions(Seq(
  ("imp_7", "ad_G", "12:21")
))

writeClicks(Seq(
  ("clk_8", "ad_G", "12:27")
))

runStreamingJoin()

Wrote 1 impressions:
  imp_7     ad=ad_G   time=12:21
Wrote 1 clicks:
  clk_8     ad=ad_G   time=12:27


26/04/05 21:50:29 WARN Column: Constructing trivially true equals predicate, 'ad_id == ad_id'. Perhaps you need to use aliases.
26/04/05 21:50:29 INFO AbstractParser$ParserCaches: EXPERIMENTAL: Query cached 0 DFA states in the parser. Total cached DFA states: 5in the parser. Driver memory: 22649241600.
26/04/05 21:50:30 INFO StreamingQueryListenerBus: Received QueryStartedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


Query started: 1120a40b-a62c-4b4d-993a-44174cbd215b


26/04/05 21:50:30 INFO DataStreamWriter: Called streaming start call back for query: 1120a40b-a62c-4b4d-993a-44174cbd215b


  eventTime.watermark: 2025-01-15T12:17:00.000Z
  stateOp: symmetricHashJoin | rows_updated=2, rows_removed=0, dropped_by_watermark=0, rows_total=7


26/04/05 21:51:26 INFO StreamingQueryListenerBus: Received QueryTerminatedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


Streaming join complete.
Query terminated: 1120a40b-a62c-4b4d-993a-44174cbd215b


### Run 4 Results
**6 rows** — the new pair (imp_7, clk_8) matched successfully.

In [0]:
showResults()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180
imp_4,clk_4,ad_D,2025-01-15T12:20:00Z,2025-01-15T12:29:00Z,540
imp_7,clk_8,ad_G,2025-01-15T12:21:00Z,2025-01-15T12:27:00Z,360
imp_5,clk_6,ad_E,2025-01-15T12:22:00Z,2025-01-15T12:25:00Z,180


In [0]:
display(spark.read.format("statestore").option("joinSide", "left").load(checkpointPath))

key,value,partition_id
List(ad_D),"List(imp_4, ad_D, 2025-01-15T12:20:00Z)",5
List(ad_G),"List(imp_7, ad_G, 2025-01-15T12:21:00Z)",16
List(ad_E),"List(imp_5, ad_E, 2025-01-15T12:22:00Z)",67


In [0]:
display(spark.read.format("statestore").option("joinSide", "right").load(checkpointPath))

key,value,partition_id
List(ad_D),"List(clk_4, ad_D, 2025-01-15T12:29:00Z)",5
List(ad_D),"List(clk_5, ad_D, 2025-01-15T12:31:00Z)",5
List(ad_G),"List(clk_8, ad_G, 2025-01-15T12:27:00Z)",16
List(ad_E),"List(clk_6, ad_E, 2025-01-15T12:25:00Z)",67


---
## Run 5: State Eviction — clk_5 Finally Removed

Remember `clk_5` (ad_D, click_time=12:31) from Run 2? It didn't match any impression (11 min > 10-min range), but Spark kept it in the right-side state store — because a future impression with `impression_time` between 12:21 and 12:31 could still arrive and match it.

Now we advance the impression watermark past 12:31 so that's no longer possible. We write an impression at **12:40**:
- **Impression watermark** advances to max(12:40) − 5 min = **12:35**
- Since 12:35 > 12:31, no future impression can have `impression_time ≤ 12:31`
- Therefore `clk_5` can never be matched → Spark **evicts it** from state

We also write a click at 12:43 for the new impression (3-min delay, within range) so we get a match.

**Expected: 1 new match (imp_8 + clk_9) → 7 total rows. clk_5 disappears from the right-side state store.**

In [0]:
writeImpressions(Seq(
  ("imp_8", "ad_H", "12:40")
))

writeClicks(Seq(
  ("clk_9", "ad_H", "12:43")
))

runStreamingJoin()

Wrote 1 impressions:
  imp_8     ad=ad_H   time=12:40
Wrote 1 clicks:
  clk_9     ad=ad_H   time=12:43


26/04/05 21:53:28 WARN Column: Constructing trivially true equals predicate, 'ad_id == ad_id'. Perhaps you need to use aliases.
26/04/05 21:53:28 INFO AbstractParser$ParserCaches: EXPERIMENTAL: Query cached 0 DFA states in the parser. Total cached DFA states: 5in the parser. Driver memory: 22649241600.
26/04/05 21:53:29 INFO StreamingQueryListenerBus: Received QueryStartedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


Query started: 1120a40b-a62c-4b4d-993a-44174cbd215b


26/04/05 21:53:29 INFO DataStreamWriter: Called streaming start call back for query: 1120a40b-a62c-4b4d-993a-44174cbd215b


  eventTime.watermark: 2025-01-15T12:17:00.000Z
  stateOp: symmetricHashJoin | rows_updated=2, rows_removed=0, dropped_by_watermark=0, rows_total=9
  eventTime.watermark: 2025-01-15T12:35:00.000Z
  stateOp: symmetricHashJoin | rows_updated=0, rows_removed=7, dropped_by_watermark=0, rows_total=2
Streaming join complete.


26/04/05 21:54:20 INFO StreamingQueryListenerBus: Received QueryTerminatedEvent for query 1120a40b-a62c-4b4d-993a-44174cbd215b, posting to listener ammonite.$sess.cmd7$Helper$$anon$1


Query terminated: 1120a40b-a62c-4b4d-993a-44174cbd215b


### Run 5 Results
**7 rows** — the new pair (imp_8, clk_9) matched. Check the right-side state store below — `clk_5` should be gone.

Updated watermarks:
- **Impression watermark** = max(12:40) − 5 min = **12:35**
- **Click watermark** = max(12:43) − 5 min = **12:38**

In [0]:
showResults()

impression_id,click_id,ad_id,impression_time,click_time,delay_seconds
imp_1,clk_1,ad_A,2025-01-15T12:00:00Z,2025-01-15T12:03:00Z,180
imp_2,clk_2,ad_B,2025-01-15T12:02:00Z,2025-01-15T12:06:00Z,240
imp_3,clk_3,ad_C,2025-01-15T12:05:00Z,2025-01-15T12:08:00Z,180
imp_4,clk_4,ad_D,2025-01-15T12:20:00Z,2025-01-15T12:29:00Z,540
imp_7,clk_8,ad_G,2025-01-15T12:21:00Z,2025-01-15T12:27:00Z,360
imp_5,clk_6,ad_E,2025-01-15T12:22:00Z,2025-01-15T12:25:00Z,180
imp_8,clk_9,ad_H,2025-01-15T12:40:00Z,2025-01-15T12:43:00Z,180


In [0]:
display(spark.read.format("statestore").option("joinSide", "left").load(checkpointPath))

key,value,partition_id
List(ad_H),"List(imp_8, ad_H, 2025-01-15T12:40:00Z)",116


In the right-side state store below, `clk_5` (ad_D, 12:31) is **no longer present** — it was evicted because the impression watermark (12:35) now guarantees no future impression can match it.

In [0]:
display(spark.read.format("statestore").option("joinSide", "right").load(checkpointPath))

key,value,partition_id
List(ad_H),"List(clk_9, ad_H, 2025-01-15T12:43:00Z)",116


In [0]:
showGlobalWatermarks()

batch_id,watermark_epoch_ms,watermark_timestamp
0,1736942400000,2025-01-15T12:00Z
1,1736942400000,2025-01-15T12:00Z
2,1736943420000,2025-01-15T12:17Z
3,1736943420000,2025-01-15T12:17Z
4,1736943420000,2025-01-15T12:17Z
5,1736943420000,2025-01-15T12:17Z
6,1736944500000,2025-01-15T12:35Z
7,1736944500000,2025-01-15T12:35Z


## Summary

| Run | What Happened | Matches? | Why |
|---|---|---|---|
| 1 | On-time impressions + clicks, all within range | **3 matches** | All data within watermark and 10-min range |
| 2 | Clicks at 9 min (in range) and 11 min (out of range) | **2 new matches** | Range condition filters out the 11-min click |
| 3 | Late impression at 12:10 + click at 12:12 | **0 new matches** | Both behind the watermark → dropped |
| 4 | Impression at 12:21 + click at 12:27 | **1 new match** | Event times still ahead of watermark → accepted |
| 5 | Impression at 12:40 advances watermark to 12:35 | **1 new match** | clk_5 evicted — impression watermark (12:35) > clk_5 time (12:31) |

### Key Takeaways

1. **Watermarks control late data.** Data with event times behind the watermark (`max_event_time - delay`) is dropped before entering state. This prevents stale data from accumulating.

2. **The range condition controls match validity.** Even if data passes the watermark, it must satisfy the time range to produce a join result.

3. **Both are needed to bound state in a join, but for different reasons.** The watermark alone filters late *input*, but cannot evict rows already *in* state — without a range condition, any future row could match any past row, so nothing is safe to remove. The range condition gives Spark the eviction rule: once `watermark > row_time + range`, that buffered row can never be matched and is evicted. Together, they bound both what enters state and how long it stays.
[info on eviction requirements in streaming joins](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#inner-joins-with-optional-watermarking)

4. **The watermark is not a wall clock — it's driven by the data.** It advances based on the maximum event time seen, not the current time. This makes it testable and deterministic.

## Cleanup
Uncomment and run the cell below to remove all demo tables and state.

In [0]:
// spark.sql(s"DROP TABLE IF EXISTS $outputTable")
// spark.sql(s"DROP TABLE IF EXISTS $impressionsTable")
// spark.sql(s"DROP TABLE IF EXISTS $clicksTable")
// spark.sql(s"DROP SCHEMA IF EXISTS $demoCatalog.$demoSchema CASCADE")
// dbutils.fs.rm(checkpointPath, recurse = true)
// println("Cleanup complete.")